In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    StorageContext,
    Settings,
    load_index_from_storage,
)
from llama_index.core.tools import QueryEngineTool
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
# from llama_index.llms.openai_like import OpenAILike

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

llm = HuggingFaceInferenceAPI(
    model_name="Qwen/Qwen2.5-Coder-32B-Instruct",
    token=HF_TOKEN,
)
# llm = OpenAILike(
#     model="Qwen/Qwen2.5-Coder-32B-Instruct",
#     api_base="https://router.huggingface.co/v1",
#     api_key=HF_TOKEN,
#     is_chat_model=True,
#     is_function_calling_model=False,  # keeps the agent on the ReAct loop, as in the course
#     context_window=32000,
#     max_tokens=1024,
#     temperature=0.1,
# )

# Embeddings run locally (no API cost), LLM runs via the Inference API
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

Settings.llm = llm
Settings.embed_model = embed_model

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

In [2]:
KNOWLEDGE_DIR = Path("knowledge")
PERSIST_DIR = Path("storage")

if PERSIST_DIR.exists():
    # Reuse the saved index: skips re-embedding on every run
    storage_context = StorageContext.from_defaults(persist_dir=str(PERSIST_DIR))
    index = load_index_from_storage(storage_context)
    print("Loaded index from disk")
else:
    documents = SimpleDirectoryReader(
        input_dir=str(KNOWLEDGE_DIR),
        required_exts=[".md"],
        recursive=True,
    ).load_data()
    print(f"Loaded {len(documents)} documents")

    index = VectorStoreIndex.from_documents(documents, show_progress=True)
    index.storage_context.persist(persist_dir=str(PERSIST_DIR))
    print("Built and saved index")

Loaded index from disk


In [3]:
query_engine = index.as_query_engine(
    llm=llm,
    similarity_top_k=4,
    response_mode="compact",
)

knowledge_tool = QueryEngineTool.from_defaults(
    query_engine=query_engine,
    name="knowledge_base",
    description=(
        "Searches my local markdown knowledge base. "
        "Use this for any question about the contents of those documents."
    ),
    return_direct=False,
)

In [4]:
agent = AgentWorkflow.from_tools_or_functions(
    [knowledge_tool],
    llm=llm,
    system_prompt=(
        "You are a helpful assistant that answers questions using the knowledge_base tool. "
        "Always search the knowledge base before answering questions about its content. "
        "If the answer is not in the knowledge base, say so clearly."
    ),
)

In [5]:
response = await agent.run("What is this knowledge base about?")
print(response)

This knowledge base is about the OutreachFlow application, providing detailed documentation for each feature within the app. It covers various aspects including architecture, data models, user flows, controllers, services, jobs, routes, credits and billing impact, external integrations, configuration settings, permissions, edge cases, testing, future work, and related documentation. The structure is organized into sections such as foundation, data acquisition, organization, sending infrastructure, outreach, supporting features, and admin & operations, ensuring comprehensive coverage of the application's functionalities.


In [6]:
response = await agent.run("Can you tell me more about credit system and how much it charge")
print(response)

The credit system involves purchasing credits through subscription plans, with each plan offering a set number of credits that reset upon payment. Charges for different actions are determined by the `CreditService`, which references configurations in `config/credits.php` and environment variables. For instance, finding a website usually costs 1 credit, and sending an email typically costs 2 credits. These costs can vary based on configuration. There's also a bonus system for website discovery, aiming to provide 30% more results than requested without extra charges, though users receive at least the number of results they paid for if the bonus isn't met.
